# Redis in low-latency & distributed trading (Python)

This notebook is a **focused companion** to [`g_redis.ipynb`](g_redis.ipynb): it connects Redis data structures and patterns to **trading-engine style** workloads—**multi-venue market data**, **buffering / replay**, **coordination**, and **order-flow helpers**—in a **distributed** system.

**Naming:** *HFT* here means **tight latency budgets** (e.g. **microseconds to low milliseconds** on the LAN) and **high throughput**, **not** nanosecond colocated tick-to-trade hardware. Redis is **single-threaded** per process and **network-bound**; it is a **coordination + fast state + streaming** layer, not a replacement for a full OMS/EMS or exchange co-lo stack.

**Requirements:** `pip install redis`, Redis at `REDIS_URL` (default `redis://localhost:6379/0`). Cells **degrade gracefully** if Redis is down.


### Contents

- [1. Where Redis helps (and where it does not)](#1-where-redis-helps-and-where-it-does-not)
- [2. Multi-venue ingestion: Streams and consumer groups](#2-multi-venue-ingestion-streams-and-consumer-groups)
- [3. Idempotency, deduplication, and sequence checkpoints](#3-idempotency-deduplication-and-sequence-checkpoints)
- [4. Time-ordered windows: sorted sets for recent ticks / events](#4-time-ordered-windows-sorted-sets-for-recent-ticks--events)
- [5. Latest snapshots: hashes for books, candles, and session state](#5-latest-snapshots-hashes-for-books-candles-and-session-state)
- [6. Live fan-out vs durability: Pub/Sub vs Streams](#6-live-fan-out-vs-durability-pubsub-vs-streams)
- [7. Order placement helpers: atomic Lua, state, and rate limits](#7-order-placement-helpers-atomic-lua-state-and-rate-limits)
- [8. Batching with pipelines and a small ops checklist](#8-batching-with-pipelines-and-a-small-ops-checklist)


## 1. Where Redis helps (and where it does not)

| Fit | Examples |
|-----|----------|
| **Buffered streams** from many connectors | Ticks, book updates, candles, alt data → **normalize** in workers |
| **Replay / recovery** of processing | **Consumer groups**, pending + **claim** after crashes |
| **Fast shared state** | Latest book snapshot, session flags, **idempotency** keys |
| **Coordination** | Leader election–style locks, **rate limits** to REST/WebSocket APIs |
| **Ephemeral fan-out** | Dashboards, alerts → **Pub/Sub** (no persistence) |

| Usually *outside* Redis on the critical path | Primary **order book** at the exchange, **regulatory** book of record, **long-term** tick storage (use object store / TSDB / warehouse) |

**Latency:** expect **sub-ms to a few ms** per round-trip over TCP to Redis unless you co-locate and tune. That is **compatible** with many **distributed** strategies; it is **not** the same as **FPGA / kernel-bypass** co-lo.


## 2. Multi-venue ingestion: Streams and consumer groups

**Pattern:** one stream per **logical channel** (e.g. venue + feed type + symbol class) or one stream per **asset** with fields `venue`, `type`, `payload`. **Consumer groups** let **N workers** share load; **XACK** commits progress; **XAUTOCLAIM** recovers stuck messages.

Trim with **`MAXLEN ~`** so memory stays bounded; keep **cold storage** elsewhere for years of history.


In [ ]:
import os
import redis

REDIS_URL = os.environ.get("REDIS_URL", "redis://localhost:6379/0")
NS = "g_redis_hft:ingest"
STREAM = NS + ":ticks"
GROUP = "normalizers"
CONSUMER = "worker-1"

try:
    r = redis.from_url(REDIS_URL, decode_responses=True)
    r.ping()
    r.delete(STREAM)

    # Simulate two venues pushing into the same stream (fields are flat strings)
    r.xadd(
        STREAM,
        {"venue": "A", "sym": "BTCUSDT", "px": "65000.5", "qty": "0.1", "ts_us": "1700000000000123"},
        maxlen=100_000,
        approximate=True,
    )
    r.xadd(
        STREAM,
        {"venue": "B", "sym": "BTCUSDT", "px": "65001.0", "qty": "0.05", "ts_us": "1700000000000456"},
        maxlen=100_000,
        approximate=True,
    )

    r.xgroup_create(STREAM, GROUP, id="0", mkstream=True)
    resp = r.xreadgroup(GROUP, CONSUMER, {STREAM: ">"}, count=10, block=500)
    for _sname, messages in resp:
        for eid, fields in messages:
            print("IN", eid, fields)
            r.xack(STREAM, GROUP, eid)

    r.delete(STREAM)
    print("Stream ingest demo OK")
except redis.ConnectionError as e:
    print("Redis not reachable:", e)
except Exception as e:
    print("Error:", e)


## 3. Idempotency, deduplication, and sequence checkpoints

**Problem:** connectors **retry**; you must not **double-apply** the same logical event. **Pattern:** key = `(venue, channel, exchange_seq)` or hash of stable fields; **`SET key 1 NX EX ttl`** — if `True`, first time; if `False`, duplicate.

**Checkpoints:** store last processed **stream ID** or **exchange sequence** in a string; update in the same **pipeline** as downstream writes when possible.


In [ ]:
import os
import redis

REDIS_URL = os.environ.get("REDIS_URL", "redis://localhost:6379/0")
NS = "g_redis_hft:idem"

try:
    r = redis.from_url(REDIS_URL, decode_responses=True)
    r.ping()

    dedup_key = NS + ":evt:A:BTCUSDT:seq:42"
    r.delete(dedup_key)

    first = r.set(dedup_key, "1", nx=True, ex=3600)
    dup = r.set(dedup_key, "1", nx=True, ex=3600)
    print("First insert:", first, "Duplicate:", dup)

    ck = NS + ":checkpoint:venueA:BTCUSDT"
    r.set(ck, "1700000000-123")
    print("Checkpoint:", r.get(ck))

    r.delete(dedup_key, ck)
    print("Idempotency demo OK")
except redis.ConnectionError as e:
    print("Redis not reachable:", e)
except Exception as e:
    print("Error:", e)


## 4. Time-ordered windows: sorted sets for recent ticks / events

**Pattern:** score = **time in µs** (or ms if enough resolution); member = **unique id** (opaque payload or `seq`). Use **`ZADD`** + **`ZREMRANGEBYSCORE`** to drop older than window, or **`ZCARD`** + trim by rank. Good for **sliding windows**, **last N minutes** of events per symbol, or **merging** venues by time order (with care on clock skew).


In [ ]:
import os
import redis

REDIS_URL = os.environ.get("REDIS_URL", "redis://localhost:6379/0")
ZKEY = "g_redis_hft:win:BTCUSDT:ticks_us"

try:
    r = redis.from_url(REDIS_URL, decode_responses=True)
    r.ping()
    r.delete(ZKEY)

    base = 1_700_000_000_000_000  # fictitious µs timestamp base
    for i in range(5):
        ts_us = base + i * 1000
        r.zadd(ZKEY, {f"tick-{i}|65000.{i}": ts_us})

    # Last 3 events in time order
    print("ZRANGE last 3:", r.zrange(ZKEY, -3, -1, withscores=True))
    # Drop older than base + 2000 µs
    r.zremrangebyscore(ZKEY, "-inf", base + 2000)
    print("After trim:", r.zrange(ZKEY, 0, -1, withscores=True))

    r.delete(ZKEY)
    print("ZSET window demo OK")
except redis.ConnectionError as e:
    print("Redis not reachable:", e)
except Exception as e:
    print("Error:", e)


## 5. Latest snapshots: hashes for books, candles, and session state

**Pattern:** one hash per **symbol** or **symbol:venue** with fields like `bid0`, `ask0`, `last_candle_1m`, `updated_us`. Readers get **O(1)** `HGETALL` for UI or **risk** snapshots. Combine with **TTL** if the key should expire when the session ends.

For **full depth**, payloads can be **JSON/text in one field** or **multiple levels** as separate fields—trade off **size** vs **partial updates**.


In [ ]:
import os
import json
import redis

REDIS_URL = os.environ.get("REDIS_URL", "redis://localhost:6379/0")
HKEY = "g_redis_hft:snap:BTCUSDT:A"

try:
    r = redis.from_url(REDIS_URL, decode_responses=True)
    r.ping()
    r.delete(HKEY)

    book = {"bids": [["65000", "1.0"], ["64999", "2.0"]], "asks": [["65001", "0.5"]]}
    r.hset(
        HKEY,
        mapping={
            "last_px": "65000.5",
            "updated_us": "1700000000000999",
            "book_json": json.dumps(book),
        },
    )
    r.expire(HKEY, 86400)
    print("HGETALL:", r.hgetall(HKEY))

    r.delete(HKEY)
    print("Snapshot hash demo OK")
except redis.ConnectionError as e:
    print("Redis not reachable:", e)
except Exception as e:
    print("Error:", e)


## 6. Live fan-out vs durability: Pub/Sub vs Streams

| Mechanism | Latency | Durability | Typical use |
|-----------|---------|------------|-------------|
| **Pub/Sub** | Very low | **None** (miss if offline) | Live **quotes** to dashboards, **alerts** |
| **Streams** | Still low | **Retained** until trimmed | **Ingest**, **replay**, **multi-consumer** pipelines |

Many systems use **both:** Streams for **truth in the pipeline**, Pub/Sub for **ephemeral** UI updates derived from the same worker.


## 7. Order placement helpers: atomic Lua, state, and rate limits

**Atomicity:** multiple reads/writes (e.g. decrement **risk budget** and record **client order id**) can be one **`EVAL`** script so another instance does not interleave.

**Rate limits:** sliding window in a **ZSET** (timestamp scores) or **INCR** with **TTL** for fixed windows—protects **REST** connectors to exchanges.

**Caveat:** the **exchange** is the source of truth for **fills**; Redis holds **intent** and **cache** state until confirmed.


In [ ]:
import os
import time
import redis

REDIS_URL = os.environ.get("REDIS_URL", "redis://localhost:6379/0")
NS = "g_redis_hft:order"

try:
    r = redis.from_url(REDIS_URL, decode_responses=True)
    r.ping()

    risk_key = NS + ":risk:BTC"
    r.set(risk_key, "100")

    lua = """
    local k = KEYS[1]
    local need = tonumber(ARGV[1])
    local cur = tonumber(redis.call('GET', k) or '0')
    if cur < need then return {0, cur} end
    redis.call('DECRBY', k, need)
    return {1, redis.call('GET', k)}
    """
    ok, left = r.eval(lua, 1, risk_key, 30)
    print("Lua risk decrement:", ok, left)

    # Simple per-second rate limit bucket (fixed window)
    rl_key = NS + ":rl:venueA:" + str(int(time.time()))
    n = r.incr(rl_key)
    if n == 1:
        r.expire(rl_key, 2)
    print("Rate count this second:", n)

    r.delete(risk_key, rl_key)
    print("Order helper demo OK")
except redis.ConnectionError as e:
    print("Redis not reachable:", e)
except Exception as e:
    print("Error:", e)


## 8. Batching with pipelines and a small ops checklist

**Pipelines:** batch many **`HGET`/`XACK`/`ZADD`** calls to cut **round-trips** when one process flushes a burst of updates.

**Checklist (short):**

- **Shard by symbol or venue** to avoid one hot key for the whole exchange.
- **Co-locate** Redis with **consumers** that chat with it constantly; watch **p99 RTT**.
- **AOF**/`appendfsync` policies vs throughput—understand **durability** if Streams are your **recovery** backbone.
- **Monitor:** stream **lag**, **XPENDING** growth, **memory**, **blocked clients**.
- **Legal/compliance:** Redis is rarely the **system of record** for orders—reconcile to **DB + exchange reports**.

For core Redis APIs, see **`g_redis.ipynb`**.


In [ ]:
import os
import redis

REDIS_URL = os.environ.get("REDIS_URL", "redis://localhost:6379/0")

try:
    r = redis.from_url(REDIS_URL, decode_responses=True)
    r.ping()
    p = r.pipeline()
    for i in range(5):
        p.set(f"g_redis_hft:pipe:{i}", str(i), ex=60)
    p.execute()
    p2 = r.pipeline()
    for i in range(5):
        p2.get(f"g_redis_hft:pipe:{i}")
    print("Pipeline GETs:", p2.execute())
    r.delete(*[f"g_redis_hft:pipe:{i}" for i in range(5)])
    print("Pipeline demo OK")
except redis.ConnectionError as e:
    print("Redis not reachable:", e)
except Exception as e:
    print("Error:", e)
